# Lesson 2: The `gds` Object and AGA Basics

**Module:** AGA Fundamentals  
**Dataset:** Movies (Actors, Movies, Genres, Users)

This notebook spins up a session, projects an `Actor-COLLABORATED-Actor` graph from your AuraDB instance, and exercises the `gds` object's API surface — `run_cypher`, `graph.project`, the three execution modes (`stream`, `mutate`, `write`), and the cleanup discipline that bookends every AGA-from-Python script.

By the end you'll have run a complete (small) workflow against AGA from Python.

## Imports

In [ ]:
import os
from datetime import timedelta
from IPython.display import display
from dotenv import load_dotenv
from graphdatascience.session import GdsSessions, AuraAPICredentials, DbmsConnectionInfo, AlgorithmCategory

## Setup

Load your `.env` credentials and create the sessions manager. These two cells are the same in every notebook in this module.

In [ ]:
load_dotenv()

client_id     = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id    = os.getenv('AURA_PROJECT_ID')
uri           = os.getenv('AURA_URI')
username      = os.getenv('AURA_USERNAME')
password      = os.getenv('AURA_PASSWORD')

In [ ]:
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id)
)
print("Sessions manager ready.")

## Estimate the session memory

Before creating a session, you can ask Aura roughly how much memory the workload will need. `sessions.estimate(...)` takes approximate node and relationship counts plus the algorithm categories you plan to run and returns the right `SessionMemory` tier.

Skip this step, and you'll either get a session too small for the projection or one larger than you need to pay for.

The workflow you're about to run projects an `Actor-COLLABORATED-Actor` graph from the Movies dataset and runs a centrality algorithm on it. That's around 37,000 nodes and well over a million relationships once projected. `CENTRALITY` category is a good estimate to start from.

In [ ]:
memory = sessions.estimate(
    node_count=37_000,
    relationship_count=500_000,
    algorithm_categories=[AlgorithmCategory.CENTRALITY],
)
print(f"Estimator recommends: {memory}")

## Create the session

With the memory tier in hand, you can create the session itself. `sessions.get_or_create(...)` uses the value you just estimated and connects the session to your AuraDB through `DbmsConnectionInfo`.

A 30-minute TTL gives you breathing room to walk through the workflow without an idle session being terminated mid-experiment.

In [ ]:
gds = sessions.get_or_create(
    session_name="workflow-demo",
    memory=memory,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password,
    ),
    ttl=timedelta(minutes=30),
)
gds.verify_connectivity()
print(f"Connected to GDS Session: {gds}")

You'll see a `Connected to GDS Session: ...` confirmation. From here on the `gds` object behaves exactly like the `GraphDataScience` instance you've used against local Neo4j or AuraDS — every method on it works the same way.

## Run Cypher against your database

Now that the session is up and pointing at your AuraDB, you can run Cypher through it. `gds.run_cypher(...)` sends the query to AuraDB and returns the result as a pandas DataFrame.

Run the code below to get a quick label-count.

In [ ]:
df = gds.run_cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY count DESC
""")
display(df)

The DataFrame has one row per node label and its count. You should see `Actor`, `Movie`, `Genre`, and `User`.

## Project the actor-collaboration graph

There is a slight nuance to projecting into an Aura Graph Analytics session from Python. `gds.graph.project(name, cypher)` takes a Cypher query that ends in `gds.graph.project.remote(...)` — the Cypher describes which subgraph to project, and the remote-projection procedure actually sends it.

The projection you're about to build is `Actor-COLLABORATED-Actor`: two actors are connected if they've appeared in at least one movie together, weighted by how many movies they share.

In [ ]:
G, result = gds.graph.project(
    "actor-collaborations",
    """
    CALL {
      MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
      WHERE a1 <> a2
      WITH a1, a2, count(m) AS collaborations
      RETURN a1 AS source, a2 AS target, collaborations
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      relationshipType: 'COLLABORATED',
      relationshipProperties: {collaborations: collaborations}
    })
    """
)
print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

`G` is your graph — from now on any time you want to interact with it, you'll refer to it as `G`.

## Stream results back to Python

With a projection ready, you can run algorithms against it. The next three cells walk through `stream`, `mutate`, and `write` against the same algorithm: **degree centrality**.

The first mode, `stream`, runs the algorithm inside the session and returns the results as a DataFrame to Python. Use it when the result *is* the output — you want to inspect it, plot it, or feed it into something else in Python.

In [ ]:
df = gds.degree.stream(G)
display(df.nlargest(10, "score"))

The DataFrame has one row per node with `nodeId` and `score` columns, sorted to the top-10 most-connected actors. `nodeId` is the session-internal ID, not the AuraDB node ID — that's why the column shows integers rather than actor names. You'll resolve back to names in a moment.

## Mutate — write into the session

The code below will run `degree` again, but this time, in `mutate` mode. Instead of returning the result to Python, it will store the score as a node property **inside the session's projection**. It won't write back to your graph in AuraDB.

In [ ]:
result = gds.degree.mutate(G, mutateProperty="rank")
print(f"Nodes processed: {result['nodePropertiesWritten']:,}")
print(f"Actor properties now: {G.node_properties('Actor')}")

The print shows how many nodes received a `rank` property, and lists the properties now available on Actor nodes in the projection. The score is queryable from inside the session, but your AuraDB is still untouched.

## Write — persist to AuraDB

`write` persists results back to AuraDB. Run the code below to add the "degree" property to your main graph in AuraDB.

In [ ]:
result = gds.degree.write(G, writeProperty="degree")
print(f"Wrote degree centrality to {result['nodePropertiesWritten']:,} nodes")

The print confirms how many nodes received the `degree` property. The session computed the value, but the property now lives in AuraDB — independent of the session.

## Verify the write

`gds.run_cypher` allows you to run arbitrary Cypher queries against your graph in AuraDB. Run the following query to see which actors received the highest `degree` score.

In [ ]:
df = gds.run_cypher("""
    MATCH (a:Actor)
    WHERE a.degree IS NOT NULL
    RETURN a.name AS actor, a.degree AS degree
    ORDER BY a.degree DESC
    LIMIT 10
""")
display(df)

The DataFrame shows the top-10 actors by number of unique collaborators. From here on, every query against this AuraDB can use `degree` like any other node property.

## Clean up

Finally, call `sessions.delete(...)` to terminate the session.

In [ ]:
sessions.delete(session_name="workflow-demo")
print("Session deleted. degree property is persisted in AuraDB.")

The session is gone, but the `degree` property is still in AuraDB.

Move on to `3_standalone.ipynb` to see how to run AGA against data that isn't in Neo4j at all.